# Week 2 Day 4: restart a dispatch
Stretch challenge from `labs/week2day4.ipynb`.

Save the dispatch with Day 3's `PostgresSaver`, restart the kernel while workers are running, then continue the same thread with `invoke(None, config)`. This notebook uses two stand-in claims and no model calls.

In [ ]:
import operator
import os
import time
from collections import Counter
from collections.abc import Callable
from datetime import datetime, timezone
from pathlib import Path
from typing import Annotated, TypedDict
from uuid import uuid4

import pandas as pd
import psycopg
from dotenv import load_dotenv
from IPython.display import display
from langgraph.checkpoint.base import BaseCheckpointSaver
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.state import CompiledStateGraph
from langgraph.types import Send
from psycopg.rows import dict_row

RUN_POSTGRES = False
THREAD_ID = "day4-restart-1"
DELAYS = {"DEMO-1": 1.0, "DEMO-2": 30.0}


class ClaimResult(TypedDict):
    claim_id: str
    started: datetime
    finished: datetime


class Scores(TypedDict):
    completed: int
    seconds_per_claim: float


class Dispatch(TypedDict, total=False):
    claims: list[str]
    results: Annotated[list[ClaimResult], operator.add]
    scores: Scores

## Build the saved dispatch
The graph keeps the first challenge's shape: list claims, work them in parallel, then score once. The checkpointer saves returned worker results. The separate event table records attempts and survives a kernel restart too.

The workers have no retry policy here. Restarting the process is the experiment; an ordinary caught error is a different event.

In [ ]:
def build_dispatcher(
    checkpointer: BaseCheckpointSaver,
    record: Callable[[str, str], None],
    delays: dict[str, float],
) -> CompiledStateGraph:
    def list_claims(state: Dispatch) -> Dispatch:
        return {"claims": list(delays)}

    def fan_out(state: Dispatch) -> list[Send]:
        return [Send("work_claim", claim_id) for claim_id in state["claims"]]

    def work_claim(claim_id: str) -> Dispatch:
        record(claim_id, "started")
        started = datetime.now(timezone.utc)
        time.sleep(delays[claim_id])
        finished = datetime.now(timezone.utc)
        record(claim_id, "finished")
        return {"results": [{"claim_id": claim_id, "started": started, "finished": finished}]}

    def score_run(state: Dispatch) -> Dispatch:
        results = state["results"]
        record("RUN", "scored")
        return {"scores": {
            "completed": len(results),
            "seconds_per_claim": sum((r["finished"] - r["started"]).total_seconds() for r in results) / len(results),
        }}

    builder = StateGraph(Dispatch)
    builder.add_node("list_claims", list_claims)
    builder.add_node("work_claim", work_claim)
    builder.add_node("score_run", score_run)
    builder.add_edge(START, "list_claims")
    builder.add_conditional_edges("list_claims", fan_out, ["work_claim"])
    builder.add_edge("work_claim", "score_run")
    builder.add_edge("score_run", END)
    return builder.compile(checkpointer=checkpointer)

## Local check
An in-memory check verifies the graph and final score without a database. It does not prove recovery across processes. The Postgres experiment below supplies that test.

In [ ]:
def check_graph() -> None:
    events: list[tuple[str, str]] = []

    def record(claim_id: str, phase: str) -> None:
        events.append((claim_id, phase))

    graph = build_dispatcher(InMemorySaver(), record, {"CHECK-1": 0.01, "CHECK-2": 0.02})
    config = {"configurable": {"thread_id": "offline-check"}, "max_concurrency": 2}
    result = graph.invoke({}, config, durability="sync")
    assert result["scores"]["completed"] == 2
    assert sorted(row["claim_id"] for row in result["results"]) == ["CHECK-1", "CHECK-2"]
    assert events.count(("RUN", "scored")) == 1
    assert not graph.get_state(config).next
    print("Local graph check passed.")


check_graph()

## Connect and rebuild
Run this cell both before the experiment and after the kernel restart. It reconnects to existing records without clearing them. Each setup has a new session ID so starts before and after the restart can be distinguished.

In [ ]:
if RUN_POSTGRES:
    ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "labs/week2day4.ipynb").is_file())
    load_dotenv(ROOT / ".env", override=True)
    connection = psycopg.connect(
        os.environ["DATABASE_URL"], autocommit=True, prepare_threshold=0,
        row_factory=dict_row, options="-c search_path=lab_day4_restart", connect_timeout=10,
    )
    connection.execute("CREATE SCHEMA IF NOT EXISTS lab_day4_restart")
    connection.execute("""CREATE TABLE IF NOT EXISTS worker_events (
        id BIGSERIAL PRIMARY KEY, thread_id TEXT NOT NULL, session_id TEXT NOT NULL,
        claim_id TEXT NOT NULL, phase TEXT NOT NULL, recorded_at TIMESTAMPTZ DEFAULT now())""")
    saver = PostgresSaver(connection)
    saver.setup()
    event_connection = psycopg.connect(
        os.environ["DATABASE_URL"], autocommit=True, row_factory=dict_row,
        options="-c search_path=lab_day4_restart", connect_timeout=10,
    )
    session_id = uuid4().hex[:8]

    def record_event(claim_id: str, phase: str) -> None:
        event_connection.execute(
            "INSERT INTO worker_events (thread_id, session_id, claim_id, phase) VALUES (%s, %s, %s, %s)",
            (THREAD_ID, session_id, claim_id, phase),
        )
        print(session_id, claim_id, phase, flush=True)

    dispatcher = build_dispatcher(saver, record_event, DELAYS)
    config = {"configurable": {"thread_id": THREAD_ID}, "max_concurrency": 2}
    print("Thread:", THREAD_ID, "Session:", session_id)
else:
    print("Postgres experiment disabled. Set RUN_POSTGRES = True to begin.")

## Start, then restart the kernel
Run the next cell on a new thread. When `DEMO-1 finished` appears, wait about two seconds, then use **Restart Kernel** while the cell is still running. The first worker has finished while the second is still running.

After restarting, rerun the cells above this heading to restore definitions and reconnect. Set `RUN_POSTGRES = True` again and keep the same thread ID. **Skip the start cell below and continue at Resume.** Do not clear checkpoints or start another dispatch on that thread.

In [ ]:
if RUN_POSTGRES:
    if dispatcher.get_state(config).values:
        raise ValueError("This thread already exists. Use Resume, or choose a new THREAD_ID for another experiment.")
    dispatcher.invoke({}, config, durability="sync")

## Resume
The new kernel has no graph state in memory. `get_state` loads it from Postgres. `invoke(None, config)` continues the saved dispatch. Passing a new input dictionary would submit new input instead.

A printed `finished` event precedes saving that worker's output. If the process stops in that gap, the worker can run again. Use the saved state and event history to inspect what actually happened.

In [ ]:
if RUN_POSTGRES:
    snapshot = dispatcher.get_state(config)
    if not snapshot.values:
        raise ValueError("No saved dispatch exists for this thread. Run the start cell first.")
    print("Next nodes:", snapshot.next)
    print("Saved results:", [row["claim_id"] for row in snapshot.values.get("results", [])])
    if snapshot.next:
        result = dispatcher.invoke(None, config, durability="sync")
    else:
        result = snapshot.values
        print("This dispatch is already complete.")
    assert sorted(row["claim_id"] for row in result["results"]) == sorted(DELAYS)
    assert result["scores"]["completed"] == len(DELAYS)
    assert not dispatcher.get_state(config).next
    print("Final score:", result["scores"])

In [ ]:
if RUN_POSTGRES:
    events = connection.execute(
        "SELECT session_id, claim_id, phase, recorded_at FROM worker_events WHERE thread_id = %s ORDER BY id",
        (THREAD_ID,),
    ).fetchall()
    display(pd.DataFrame(events))
    starts = Counter(row["claim_id"] for row in events if row["phase"] == "started")
    print("Worker starts:", dict(sorted(starts.items())))
    print("Claims worked more than once:", sorted(claim_id for claim_id, count in starts.items() if count > 1))
    assert sum(row["phase"] == "scored" for row in events) == 1
    event_connection.close()
    connection.close()